# AI Ethics, Bias & Fairness
Week 9 Task 4 — reproducible fairness-aware ML pipeline.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from src.fairness_pipeline import group_metrics, demographic_parity_difference, equal_opportunity_difference, group_threshold_predictions


## 1. Generate a synthetic dataset
The protected group is intentionally associated with a modest outcome disparity to make fairness analysis meaningful.

In [ ]:
rng = np.random.default_rng(42)
n = 1200
group = rng.choice(['A', 'B'], size=n, p=[0.6, 0.4])
experience = rng.normal(5, 2, n).clip(0, 12)
education = rng.normal(70, 12, n).clip(30, 100)
interview = rng.normal(72, 13, n).clip(20, 100)
group_effect = np.where(group == 'A', 0.35, -0.35)
logit = 0.45*experience + 0.035*education + 0.04*interview + group_effect - 7.5
prob = 1/(1+np.exp(-logit))
hired = rng.binomial(1, prob)
df = pd.DataFrame({'experience_years': experience, 'education_score': education, 'interview_score': interview, 'group': group, 'hired': hired})
df.head()


## 2. Train/test split and baseline model

In [ ]:
X = df[['experience_years', 'education_score', 'interview_score']]
y = df['hired']
g = df['group']
X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(X, y, g, test_size=0.25, random_state=42, stratify=y)
model = Pipeline([('scale', StandardScaler()), ('clf', LogisticRegression(random_state=42))])
model.fit(X_train, y_train)
baseline_pred = model.predict(X_test)
print(classification_report(y_test, baseline_pred))


## 3. Group-level fairness analysis

In [ ]:
baseline_metrics = group_metrics(y_test.to_numpy(), baseline_pred, g_test.to_numpy())
display(baseline_metrics)
print('Demographic parity difference:', demographic_parity_difference(baseline_metrics))
print('Equal opportunity difference:', equal_opportunity_difference(baseline_metrics))


## 4. Fairness-aware threshold adjustment

In [ ]:
probs = model.predict_proba(X_test)[:, 1]
# Educational post-processing: choose thresholds that reduce selection-rate disparity.
thresholds = {'A': 0.50, 'B': 0.45}
mitigated_pred = group_threshold_predictions(probs, g_test.to_numpy(), thresholds)
mitigated_metrics = group_metrics(y_test.to_numpy(), mitigated_pred, g_test.to_numpy())
display(mitigated_metrics)
print('Baseline accuracy:', accuracy_score(y_test, baseline_pred))
print('Mitigated accuracy:', accuracy_score(y_test, mitigated_pred))
print('Mitigated demographic parity difference:', demographic_parity_difference(mitigated_metrics))
print('Mitigated equal opportunity difference:', equal_opportunity_difference(mitigated_metrics))


## 5. Interpretation
Compare aggregate performance with group-level metrics. A mitigation strategy is useful only when its trade-offs are acceptable for the application. In high-impact domains, fairness metrics should be selected with domain experts, legal review, stakeholder input, and documented harm analysis.